<a href="https://colab.research.google.com/github/subhanalisha/Practical-AI/blob/main/RAG_for_biography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain langchain-community langchain-huggingface chromadb requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [2]:
import os
# Force upgrade langchain-core to resolve version mismatch with community
!pip install -qU langchain-core

print("Done! Please go to 'Runtime' -> 'Restart session' before continuing.")

Done! Please go to 'Runtime' -> 'Restart session' before continuing.


In [4]:
# 1. Install missing dependency zstd
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start Ollama server in the background
import subprocess
import time
import os

# Check if ollama is in path and start server
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)

# Give the server a moment to initialize
time.sleep(10)

# 4. Pull the model
!ollama pull gemma2:2b

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,955 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,084 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRele

In [7]:
!pip install -qU langchain-ollama
from langchain_ollama import OllamaLLM

# Note: Switched to gemma2:2b as it is a highly stable small model available on Ollama
llm = OllamaLLM(model="gemma2:2b", temperature=0.1)

In [ ]:
import os
import requests
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
# from langchain_community.llms import Ollama  # Or swap with your preferred LLM provider

def download_and_prepare_document():
    """
    Downloads Nikola Tesla's Autobiography from Project Gutenberg
    and simulates the raw document parsing step.
    """
    url = "https://www.gutenberg.org/cache/epub/52344/pg52344.txt"
    print("📥 Downloading 'My Inventions' by Nikola Tesla...")
    response = requests.get(url)
    response.raise_for_status()

    raw_text = response.text

    # Clean up Project Gutenberg header/footer noise to isolate the biography context
    start_marker = "MY INVENTIONS"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

    start_idx = raw_text.find(start_marker)
    end_idx = raw_text.find(end_marker)

    if start_idx != -1 and end_idx != -1:
        biography_text = raw_text[start_idx:end_idx].strip()
    else:
        biography_text = raw_text.strip()

    print(f"✅ Ingestion complete. Total characters parsed: {len(biography_text)}")
    return biography_text

def build_rag_pipeline(biography_text):
    """
    Executes Chunking, Embedding, and Vector Storage.
    """
    # 1. Advanced Custom Chunking Strategy (The Production Standard)
    # Using a 600 character size with a 100 character overlapping structural bridge
    print("\n✂️ Chunking text using RecursiveCharacterTextSplitter...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )

    # Create document objects with metadata enrichment
    chunks = text_splitter.create_documents(
        texts=[biography_text],
        metadatas=[{"source": "Nikola_Tesla_Autobiography.txt"}]
    )
    print(f"✅ Generated {len(chunks)} contextual text chunks.")

    # 2. Vector Indexing Engine
    # Using 'all-mpnet-base-v2' maps sentences into a high-quality 768-dimensional space
    print("\n🧠 Generating embedding matrices using HuggingFace...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # 3. Permanent / Local Storage Layers
    print("💾 Indexing embedding vectors into ChromaDB vector store...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./tesla_vector_db"
    )
    print("✅ Vector database configured successfully.")
    return vector_store

def execute_rag_query(vector_store, user_query, llm):
    """
    Executes context retrieval and structured prompt augmentation.
    """
    print(f"\n🔍 Searching for contexts matching query: '{user_query}'...")

    # Retrieve the top 3 most semantically similar chunks (k=3)
    retrieved_docs = vector_store.similarity_search(user_query, k=3)

    # Consolidate text segments into a single context window
    context_window = "\n---\n".join([doc.page_content for doc in retrieved_docs])

    # Enforce strict prompt grounding boundaries to combat hallucination failures
    system_prompt = f"""
    You are an AI assistant tasked with answering questions strictly using the provided context blocks.

    CRITICAL INSTRUCTIONAL RULE:
    - Answer the question based ONLY on the attached context sections below.
    - If the context does not contain the answer, reply exactly with: "I do not know based on the provided documents."
    - Do NOT invent facts or extrapolate from your pre-training weights.

    CONTEXT SECTIONS:
    {context_window}

    USER QUESTION:
    {user_query}

    GROUNDED RESPONSE:
    """

    # 4. LLM Generation Layer (Configured here for local inference via Ollama)
    print("🤖 Invoking generation engine...")
    try:
        # Initializing local model (e.g., llama3, gemma, or mistral)
        # For OpenAI/Anthropic, switch to ChatOpenAI(api_key=...) or ChatAnthropic()
        #llm = Ollama(model="llama3", temperature=0.0)
        response = llm.invoke(system_prompt)
        return response
    except Exception as e:
        print(f"⚠️ Local LLM execution skipped or failed: {e}")
        print("\n💡 Tip: Since the LLM step failed, here is the exact context retrieved from your Vector DB:")
        return context_window

# --- Execution Entry Point ---
if __name__ == "__main__":
    # Step 1: Data Ingestion
    biography_corpus = download_and_prepare_document()

    # Step 2: Build Pipeline (Splitters -> Embeddings -> DB)
    vector_db = build_rag_pipeline(biography_corpus)

    # Step 3: Test Verification Queries
    # Test Query A: Fact checked directly from Chapter 2
    query_a = "Describe the specific vision Tesla had in a park in Budapest that led to the invention of the rotating magnetic field."
    answer_a = execute_rag_query(vector_db, query_a, llm)
    print(f"\n[Answer to Query A]:\n{answer_a}")

    # Test Query B: Out-of-syllabus adversarial validation check
    query_b = "What are the core technical advantages of using PyTorch over TensorFlow for training LLMs?"
    answer_b = execute_rag_query(vector_db, query_b, llm)
    print(f"\n[Answer to Query B]:\n{answer_b}")

📥 Downloading 'My Inventions' by Nikola Tesla...
✅ Ingestion complete. Total characters parsed: 588873

✂️ Chunking text using RecursiveCharacterTextSplitter...
✅ Generated 1186 contextual text chunks.

🧠 Generating embedding matrices using HuggingFace...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


💾 Indexing embedding vectors into ChromaDB vector store...
